# 06 · Transformata Fouriera i filtrowanie sygnałów

**Kurs:** Praktyczna Analiza Danych Eye-Tracking i EEG w Pythonie

Każdy pomiar zapisywany w czasie – położenie oka, napięcie na elektrodzie, tętno ze smartwatcha, dźwięk z mikrofonu – to **sygnał**: ciąg liczb zapisywanych jedna po drugiej. Na tych zajęciach poznamy dwa narzędzia, których używa się przy niemal każdym sygnale:

* **transformatę Fouriera** – sprawdza, **z jakich rytmów (częstotliwości)** składa się sygnał,
* **filtry** – **usuwają** z sygnału rytmy, które przeszkadzają, a zostawiają te, które nas interesują.

Te dwa narzędzia idą w parze. Dopiero gdy zobaczymy, z jakich częstotliwości składa się sygnał, możemy zdecydować, co odfiltrować – i sprawdzić, czy filtr zadziałał.

**Plan zajęć**

*Część I · Transformata Fouriera*

1. Sygnał cyfrowy i sinusoida
2. Sygnał jako suma sinusoid
3. Widmo: transformata Fouriera
4. Widmo widzi to, czego nie widać w czasie
5. Częstotliwość próbkowania: ile próbek na sekundę?

*Część II · Filtrowanie*

6. Filtr = wymazywanie części widma
7. Cztery rodzaje filtrów
8. Jak dobrać filtr?
9. Granica filtra nie jest ostra

*Na koniec:* podsumowanie, słowniczek i trzy zadania, w których odczytamy tętno z nagrania ze smartwatcha.

**Dane:** wszystkie sygnały tworzymy sami w Pythonie. Dzięki temu zawsze wiemy, co jest w środku, i możemy sprawdzić, czy nasze narzędzia pokazują prawdę. Ten notebook nie potrzebuje danych EEG ani wyników innych notebooków.

**Szacowany czas:** ok. 90 minut.

> **Jak pracować z tym notebookiem**
>
> * Uruchamiaj komórki **po kolei** klawiszami **Shift + Enter**. Każda komórka działa od razu – także te z ćwiczeniami.
> * **✏️ Ćwiczenie** to krótkie zadanie w trakcie notebooka. Zwykle wystarczy **zmienić jedną liczbę** w miejscu oznaczonym `✏️` i uruchomić komórkę jeszcze raz.
> * Odpowiedź wpisujesz zamiast słowa `None`, np. `liczba_gorek = 5`. Odpowiedź literową wpisz w cudzysłowie, np. `odpowiedz = "B"`.
> * Tekst po znaku `#` to **komentarz** – Python go pomija, jest tylko dla Ciebie.
> * W Pythonie ułamki zapisujemy z **kropką**: `0.5`, a nie `0,5`.
> * Nie musisz rozumieć każdej linijki w funkcjach pomocniczych. Ważne jest, **co robią** – zawsze jest to opisane obok.
> * Coś nie działa? Kliknij **Restart** na pasku nad notebookiem, a potem **Run All**.

## 0. Przygotowanie

Wczytujemy biblioteki:

* **NumPy** – obliczenia na tablicach liczb (w tym transformata Fouriera),
* **Matplotlib** – wykresy,
* **SciPy** – gotowe filtry (przydadzą się w części II),
* **IPython** – odtwarzanie dźwięku w notebooku.

In [ ]:
import matplotlib.pyplot as plt              # rysowanie wykresów
import numpy as np                           # obliczenia na tablicach liczb, transformata Fouriera
from IPython.display import Audio, display   # odtwarzanie dźwięku w notebooku
from scipy import signal                     # gotowe filtry (część II)

# Kolory na wykresach – czytelne także dla osób z zaburzeniami widzenia barw
NIEBIESKI, POMARANCZOWY, ZIELONY, SZARY = "#2a78d6", "#eb6834", "#1baf7a", "#a0a0a0"

# Wygląd wykresów: domyślny rozmiar i delikatna siatka w tle
plt.rcParams.update({"figure.figsize": (12, 3.5), "axes.grid": True, "grid.alpha": 0.3})

print("Gotowe – biblioteki wczytane.")

# Część I · Transformata Fouriera

## 1. Sygnał cyfrowy i sinusoida

### Sygnał cyfrowy to liczby zapisywane w równych odstępach czasu

Komputer nie zapisuje sygnału w sposób ciągły. Mierzy go co pewien stały odstęp czasu – każdy taki pomiar to **próbka**. Liczba próbek na sekundę to **częstotliwość próbkowania**. Oznaczamy ją `fs` (od ang. *sampling frequency*) i podajemy w hercach (Hz).

Przykład: `fs = 100` to 100 próbek na sekundę, czyli jedna próbka co 1/100 s = 0,01 s.

Zaczynamy od **osi czasu**, czyli listy chwil, w których „robimy pomiar”.

In [ ]:
fs = 100                       # częstotliwość próbkowania: 100 próbek na sekundę
t = np.arange(0, 1, 1 / fs)    # chwile pomiaru: od 0 s do 1 s (bez 1), co 1/fs = 0,01 s

print("Liczba próbek:", len(t))
print("Pierwsze 5 chwil pomiaru [s]:", t[:5])

### Sinusoida – najprostszy rytm

Sinusoida to fala, która powtarza się regularnie. Opisują ją dwie liczby:

* **częstotliwość** `f` – ile pełnych fal (cykli) mieści się w **jednej sekundzie**. Jednostka: **herc (Hz)**. 3 Hz = 3 fale na sekundę.
* **amplituda** `A` – jak wysoko sięga fala, licząc od zera (wysokość „górki”).

W Pythonie sinusoidę zapisujemy wzorem `A * np.sin(2 * np.pi * f * t)`. Nie musisz go rozumieć matematycznie – wystarczy wiedzieć, gdzie wpisać `A`, a gdzie `f`.

In [ ]:
f = 3    # częstotliwość: 3 fale na sekundę (3 Hz)
A = 1    # amplituda: fala sięga od -1 do +1

sinusoida = A * np.sin(2 * np.pi * f * t)   # wartość fali w każdej chwili osi czasu t

fig, ax = plt.subplots()
ax.plot(t, sinusoida, "o-", color=NIEBIESKI, ms=3)   # "o-" = kropki (próbki) połączone linią
ax.set(xlabel="Czas [s]", ylabel="Wartość sygnału",
       title=f"Sinusoida: f = {f} Hz, A = {A}  (każda kropka to jedna próbka)")
plt.show()

W ciągu 1 sekundy są dokładnie **3 górki** – bo częstotliwość wynosi 3 Hz. Najwyższy punkt ma wartość **1** – to amplituda.

### ✏️ Ćwiczenie 1. Zmień rytm

1. W komórce poniżej zmień częstotliwość na **5 Hz**, a amplitudę na **2**.
2. Uruchom komórkę (**Shift + Enter**).
3. Policz górki na wykresie i wpisz ich liczbę zamiast `None`.

In [ ]:
# ✏️ Ćwiczenie 1
f = 3    # ✏️ zmień na 5
A = 1    # ✏️ zmień na 2

sinusoida = A * np.sin(2 * np.pi * f * t)   # budujemy sinusoidę (ten sam wzór co wyżej)

fig, ax = plt.subplots()
ax.plot(t, sinusoida, "o-", color=NIEBIESKI, ms=3)
ax.set(xlabel="Czas [s]", ylabel="Wartość sygnału", title=f"Ćwiczenie 1: f = {f} Hz, A = {A}")
plt.show()

liczba_gorek = None   # ✏️ wpisz, ile górek widzisz w ciągu 1 sekundy
print("Ćwiczenie 1 – liczba górek:", liczba_gorek)

### ✏️ Ćwiczenie 2. Tajemnicza sinusoida

Komputer wylosował częstotliwość i amplitudę sinusoidy. **Nie wypisuj wylosowanych liczb** – odczytaj je z wykresu:

* częstotliwość = liczba górek w ciągu 1 sekundy,
* amplituda = wysokość najwyższej górki.

In [ ]:
# ✏️ Ćwiczenie 2
losowanie = np.random.default_rng(5)   # generator liczb losowych; ziarno 5 = za każdym razem ten sam „los”
f_tajna = losowanie.integers(2, 9)     # losowa częstotliwość: od 2 do 8 Hz
A_tajna = losowanie.integers(1, 5)     # losowa amplituda: od 1 do 4
tajemnicza = A_tajna * np.sin(2 * np.pi * f_tajna * t)

fig, ax = plt.subplots()
ax.plot(t, tajemnicza, color=POMARANCZOWY)
ax.set(xlabel="Czas [s]", ylabel="Wartość sygnału", title="Ćwiczenie 2: tajemnicza sinusoida")
plt.show()

odp_f = None   # ✏️ częstotliwość w Hz (liczba górek w 1 s)
odp_A = None   # ✏️ amplituda (wysokość górki)
print(f"Ćwiczenie 2: f = {odp_f} Hz, A = {odp_A}")

## 2. Sygnał jako suma sinusoid

Prawdziwe sygnały rzadko są pojedynczą sinusoidą. Najczęściej to **mieszanka** wielu rytmów naraz – wolnych i szybkich, silnych i słabych. Zobaczmy, co się stanie, gdy dodamy do siebie dwie sinusoidy:

* **wolną:** 2 Hz, amplituda 1,
* **szybką:** 10 Hz, amplituda 0,5.

In [ ]:
fs = 100                      # 100 próbek na sekundę
t = np.arange(0, 1, 1 / fs)   # 1 sekunda nagrania

wolna = 1.0 * np.sin(2 * np.pi * 2 * t)     # 2 Hz, amplituda 1
szybka = 0.5 * np.sin(2 * np.pi * 10 * t)   # 10 Hz, amplituda 0,5
suma = wolna + szybka                       # dodajemy próbka po próbce

# Trzy wykresy jeden pod drugim: wspólna oś czasu (sharex) i wspólna skala w pionie (sharey)
fig, osie = plt.subplots(3, 1, figsize=(12, 6.5), sharex=True, sharey=True)
osie[0].plot(t, wolna, color=NIEBIESKI)
osie[0].set_title("Wolna fala: 2 Hz, amplituda 1")
osie[1].plot(t, szybka, color=POMARANCZOWY)
osie[1].set_title("Szybka fala: 10 Hz, amplituda 0,5")
osie[2].plot(t, suma, color="black")
osie[2].set_title("Suma: wolna + szybka")
osie[2].set_xlabel("Czas [s]")
plt.tight_layout()
plt.show()

Suma wygląda jak wolna fala, po której „jeżdżą” małe, szybkie ząbki. Przy dwóch składnikach da się je jeszcze rozpoznać okiem. Przy pięciu albo dziesięciu składnikach i odrobinie szumu – już nie. Potrzebujemy narzędzia, które **rozłoży sygnał na składniki**. Tym narzędziem jest transformata Fouriera.

> 💡 **Najważniejsza idea tych zajęć:** (prawie) każdy sygnał da się zapisać jako **sumę sinusoid** o różnych częstotliwościach i amplitudach. Odkrył to francuski matematyk Joseph Fourier na początku XIX wieku.

### ✏️ Ćwiczenie 3. Ząbki na fali

Spójrz na dolny wykres (suma). Ile małych ząbków (górek szybkiej fali) przypada na **jedną** dużą falę? Wskazówka: wolna fala ma 2 Hz, a szybka 10 Hz.

In [ ]:
# ✏️ Ćwiczenie 3
zabki_na_fale = None   # ✏️ wpisz, ile ząbków przypada na jedną dużą falę
print("Ćwiczenie 3 – ząbków na jedną dużą falę:", zabki_na_fale)

## 3. Widmo: transformata Fouriera

Transformata Fouriera działa jak **odczytanie przepisu z gotowej potrawy**. Patrzymy na sygnał („potrawę”) i dostajemy listę składników: *jakie częstotliwości są w środku i ile jest każdej z nich*.

Wynik nazywamy **widmem** (ang. *spectrum*). Widmo to wykres, na którym:

* oś pozioma to **częstotliwość** [Hz] – a **nie czas**!
* oś pionowa to **amplituda** – jak silny jest dany rytm.

Każda sinusoida w sygnale daje w widmie **jeden pik** (pionową kreskę). **Położenie piku = częstotliwość, wysokość piku = amplituda.**

Komputer liczy to algorytmem **FFT** (*Fast Fourier Transform*, szybka transformata Fouriera) – w NumPy to funkcja `np.fft.rfft`. Policzmy widmo naszej sumy krok po kroku.

In [ ]:
N = len(suma)                                   # liczba próbek w sygnale (tu 100)

wynik_fft = np.fft.rfft(suma)                   # FFT: dla każdej częstotliwości jedna liczba (zespolona)
amplitudy = np.abs(wynik_fft) * 2 / N           # np.abs → siła rytmu; * 2 / N → wysokość piku = amplituda sinusoidy
czestotliwosci = np.fft.rfftfreq(N, d=1 / fs)   # jakiej częstotliwości [Hz] odpowiada każda z tych liczb

print("Częstotliwości [Hz]:", czestotliwosci[:12], "…")
print("Amplitudy:          ", np.round(amplitudy[:12], 2), "…")

Prawie wszystkie amplitudy są równe 0 – oprócz dwóch: przy **2 Hz (1,0)** i przy **10 Hz (0,5)**. To dokładnie te dwa składniki, które dodaliśmy w sekcji 2!

> Funkcja `np.fft.rfft` zwraca **liczby zespolone**. Każda niesie dwie informacje: siłę rytmu (amplitudę) i jego przesunięcie w czasie (fazę). Na tych zajęciach interesuje nas tylko amplituda, dlatego bierzemy wartość bezwzględną `np.abs`. Liczb zespolonych nie musisz znać.

Narysujmy sygnał i jego widmo obok siebie.

In [ ]:
fig, (ax_czas, ax_widmo) = plt.subplots(1, 2, figsize=(13, 3.5))   # dwa wykresy obok siebie

# Lewy wykres: sygnał w czasie
ax_czas.plot(t, suma, color="black")
ax_czas.set(xlabel="Czas [s]", ylabel="Wartość sygnału", title="Sygnał w czasie")

# Prawy wykres: widmo – każda częstotliwość to pionowa kreska o wysokości równej amplitudzie
ax_widmo.vlines(czestotliwosci, 0, amplitudy, color=POMARANCZOWY, lw=3)
ax_widmo.set(xlabel="Częstotliwość [Hz]", ylabel="Amplituda", title="Widmo (transformata Fouriera)")

plt.tight_layout()
plt.show()

Te same informacje, dwa spojrzenia:

* **w czasie** widzimy, *jak sygnał się zmienia*,
* **w widmie** widzimy, *z czego się składa*: pik przy 2 Hz o wysokości 1 i pik przy 10 Hz o wysokości 0,5.

Widmo kończy się na 50 Hz – dlaczego akurat tam, wyjaśni sekcja 5.

### Trzy funkcje pomocnicze

Żeby nie przepisywać w kółko tych samych linijek, zamykamy je w funkcjach. Od teraz będziemy używać:

* `policz_widmo(sygnal, fs)` – zwraca częstotliwości i amplitudy,
* `amplituda_przy(sygnal, fs, czestotliwosc)` – zwraca wysokość widma przy jednej wybranej częstotliwości,
* `rysuj_sygnal_i_widmo(t, sygnal, fs, tytul)` – rysuje sygnał w czasie i jego widmo obok siebie.

In [ ]:
def policz_widmo(sygnal, fs):
    """Zwraca dwie tablice: częstotliwości [Hz] i amplitudy (widmo amplitudowe sygnału)."""
    N = len(sygnal)                                   # liczba próbek
    amplitudy = np.abs(np.fft.rfft(sygnal)) * 2 / N   # FFT i skalowanie: wysokość piku = amplituda sinusoidy
    amplitudy[0] = amplitudy[0] / 2                   # szczegół techniczny: punkt 0 Hz (średnia sygnału) ma inną skalę
    czestotliwosci = np.fft.rfftfreq(N, d=1 / fs)     # oś częstotliwości [Hz]
    return czestotliwosci, amplitudy


def amplituda_przy(sygnal, fs, czestotliwosc):
    """Zwraca wysokość widma przy wybranej częstotliwości [Hz]."""
    czestotliwosci, amplitudy = policz_widmo(sygnal, fs)
    indeks = np.argmin(np.abs(czestotliwosci - czestotliwosc))   # numer punktu widma najbliższego tej częstotliwości
    return amplitudy[indeks]


def rysuj_sygnal_i_widmo(t, sygnal, fs, tytul="", fmax=None, tmax=None):
    """Rysuje sygnał w czasie (po lewej) i jego widmo (po prawej).

    fmax – do jakiej częstotliwości [Hz] pokazać widmo (domyślnie do końca),
    tmax – do jakiej chwili [s] pokazać sygnał (domyślnie całość).
    """
    czestotliwosci, amplitudy = policz_widmo(sygnal, fs)
    fig, (ax_czas, ax_widmo) = plt.subplots(1, 2, figsize=(13, 3.5))
    ax_czas.plot(t, sygnal, color="black", lw=1)
    ax_czas.set(xlabel="Czas [s]", ylabel="Wartość sygnału", title="Sygnał w czasie",
                xlim=(0, tmax if tmax else t[-1]))
    ax_widmo.vlines(czestotliwosci, 0, amplitudy, color=POMARANCZOWY, lw=2)   # pionowe kreski = widmo
    ax_widmo.set(xlabel="Częstotliwość [Hz]", ylabel="Amplituda", title="Widmo",
                 xlim=(0, fmax if fmax else fs / 2), ylim=(0, None))
    fig.suptitle(tytul, fontweight="bold")
    plt.tight_layout()
    plt.show()


# Test: ta sama suma co wyżej – powinny być piki przy 2 Hz (wysokość 1) i przy 10 Hz (wysokość 0,5)
rysuj_sygnal_i_widmo(t, suma, fs, tytul="Suma 2 Hz + 10 Hz", fmax=20)
print(f"Amplituda przy 10 Hz: {amplituda_przy(suma, fs, 10):.2f}")

### ✏️ Ćwiczenie 4. Sygnał z przepisu

Zbuduj sygnał z trzech składników i sprawdź, czy widmo pokaże je poprawnie.

| Składnik | Częstotliwość | Amplituda |
|---|---|---|
| 1 | 5 Hz | 2 |
| 2 | 20 Hz | 1 |
| 3 | 35 Hz | 0,5 |

Pierwszy składnik jest gotowy – to wzór do naśladowania. Uzupełnij dwa pozostałe (zamiast zer), uruchom komórkę i odczytaj z widma **wysokość piku przy 35 Hz**.

In [ ]:
# ✏️ Ćwiczenie 4
fs = 100                      # 100 próbek na sekundę
t = np.arange(0, 1, 1 / fs)   # 1 sekunda

skladnik_1 = 2 * np.sin(2 * np.pi * 5 * t)   # 5 Hz, amplituda 2 (gotowe – wzór do naśladowania)
skladnik_2 = 0 * np.sin(2 * np.pi * 0 * t)   # ✏️ wpisz amplitudę 1 i częstotliwość 20
skladnik_3 = 0 * np.sin(2 * np.pi * 0 * t)   # ✏️ wpisz amplitudę 0.5 (z kropką!) i częstotliwość 35
sygnal_z_przepisu = skladnik_1 + skladnik_2 + skladnik_3

rysuj_sygnal_i_widmo(t, sygnal_z_przepisu, fs, tytul="Ćwiczenie 4: sygnał z przepisu")

wysokosc_35 = None   # ✏️ odczytaj z widma wysokość piku przy 35 Hz
print("Ćwiczenie 4 – wysokość piku przy 35 Hz:", wysokosc_35)

### ✏️ Ćwiczenie 5. Odczytaj przepis

Teraz odwrotnie: komputer wylosował sygnał z **trzech** sinusoid. Spójrz na jego widmo i odczytaj przepis: trzy częstotliwości i trzy amplitudy. (Nie wypisuj wylosowanych liczb – odczytaj je z wykresu!)

In [ ]:
# ✏️ Ćwiczenie 5
fs = 100                      # 100 próbek na sekundę
t = np.arange(0, 1, 1 / fs)   # 1 sekunda

losowanie = np.random.default_rng(25)                                          # ziarno 25 = zawsze ten sam „los”
tajne_f = np.sort(losowanie.choice(np.arange(3, 41), size=3, replace=False))   # 3 różne częstotliwości (3–40 Hz)
tajne_A = losowanie.choice([0.5, 1.0, 1.5, 2.0], size=3)                       # 3 amplitudy

# Suma trzech sinusoid o wylosowanych częstotliwościach i amplitudach
tajny_sygnal = (tajne_A[0] * np.sin(2 * np.pi * tajne_f[0] * t)
                + tajne_A[1] * np.sin(2 * np.pi * tajne_f[1] * t)
                + tajne_A[2] * np.sin(2 * np.pi * tajne_f[2] * t))

rysuj_sygnal_i_widmo(t, tajny_sygnal, fs, tytul="Ćwiczenie 5: tajemniczy sygnał")

odp_czestotliwosci = [None, None, None]   # ✏️ trzy częstotliwości [Hz], od najmniejszej
odp_amplitudy = [None, None, None]        # ✏️ trzy amplitudy – w tej samej kolejności
print("Ćwiczenie 5 – częstotliwości:", odp_czestotliwosci, "| amplitudy:", odp_amplitudy)

### 🔊 Przykład z życia: dźwięk

Dźwięk to też sygnał – drgania powietrza. **Wysokość dźwięku to jego częstotliwość:** dźwięk „a”, do którego stroi się instrumenty, ma 440 Hz. Zbudujmy dźwięk z trzech tonów: niskiego (220 Hz), średniego (440 Hz) i wysokiego (880 Hz).

Kliknij ▶ pod wykresem, żeby posłuchać (najlepiej w słuchawkach i niezbyt głośno).

In [ ]:
fs_audio = 8000                             # dźwięk wymaga wielu próbek: 8000 na sekundę
t_audio = np.arange(0, 1.5, 1 / fs_audio)   # 1,5 sekundy dźwięku

niski = 1.0 * np.sin(2 * np.pi * 220 * t_audio)    # ton niski: 220 Hz
sredni = 0.6 * np.sin(2 * np.pi * 440 * t_audio)   # ton średni: 440 Hz (dźwięk „a”)
wysoki = 0.4 * np.sin(2 * np.pi * 880 * t_audio)   # ton wysoki: 880 Hz
akord = niski + sredni + wysoki                    # trzy tony naraz

# W czasie pokazujemy tylko pierwsze 20 ms – inaczej fale zlałyby się w jedną plamę
rysuj_sygnal_i_widmo(t_audio, akord, fs_audio, tytul="Dźwięk z trzech tonów", fmax=1200, tmax=0.02)
display(Audio(akord, rate=fs_audio))   # odtwarzacz dźwięku – kliknij ▶

Widmo pokazuje trzy piki – trzy tony. Na tej samej zasadzie działają aplikacje do strojenia gitary: liczą widmo dźwięku z mikrofonu i pokazują, gdzie leży najsilniejszy pik.

## 4. Widmo widzi to, czego nie widać w czasie

W prawdziwych pomiarach zawsze jest **szum** – drobne, przypadkowe zakłócenia. Ukryjmy sinusoidę 10 Hz (amplituda 1) w silnym szumie i sprawdźmy, czy da się ją znaleźć.

In [ ]:
fs = 250                                  # 250 próbek na sekundę
czas_nagrania = 2                         # długość nagrania [s]
t = np.arange(0, czas_nagrania, 1 / fs)   # oś czasu

losowanie = np.random.default_rng(1)              # stałe ziarno – za każdym razem ten sam szum
ukryta = 1.0 * np.sin(2 * np.pi * 10 * t)         # sinusoida 10 Hz, amplituda 1
szum = 2.0 * losowanie.standard_normal(len(t))    # losowy szum, ok. 2 razy silniejszy od sinusoidy
nagranie = ukryta + szum

rysuj_sygnal_i_widmo(t, nagranie, fs, tytul="Sinusoida 10 Hz ukryta w szumie", fmax=60)

W czasie widać tylko „bałagan” – sinusoidy nie da się dostrzec. W widmie jest inaczej:

* szum rozkłada się cienko na **wszystkie** częstotliwości – tworzy niską „trawę”,
* sinusoida skupia całą swoją siłę w **jednym** miejscu – przy 10 Hz wyrasta wyraźny pik.

Dlatego widmo jest tak przydatne: pozwala znaleźć rytmy, których w surowym sygnale nie widać. Może to być rytm serca w zapisie ze smartwatcha albo zakłócenie z sieci elektrycznej w nagraniu z laboratorium.

### ✏️ Ćwiczenie 6. Dłuższe nagranie

Zmień długość nagrania z **2 s** na **10 s** i uruchom komórkę. Co dzieje się z „trawą” szumu w widmie?

* **A** – szum rośnie i zasłania pik,
* **B** – szum maleje, a pik przy 10 Hz zostaje (ma wysokość ok. 1),
* **C** – pik przy 10 Hz znika.

In [ ]:
# ✏️ Ćwiczenie 6
fs = 250                                  # 250 próbek na sekundę
czas_nagrania = 2                         # ✏️ zmień na 10
t = np.arange(0, czas_nagrania, 1 / fs)   # oś czasu

losowanie = np.random.default_rng(1)              # stałe ziarno – za każdym razem ten sam szum
ukryta = 1.0 * np.sin(2 * np.pi * 10 * t)         # sinusoida 10 Hz, amplituda 1
szum = 2.0 * losowanie.standard_normal(len(t))    # szum tej samej siły co wcześniej
nagranie = ukryta + szum

rysuj_sygnal_i_widmo(t, nagranie, fs, tytul=f"Ćwiczenie 6: nagranie {czas_nagrania} s", fmax=60)

odpowiedz_6 = None   # ✏️ wpisz "A", "B" albo "C"
print("Ćwiczenie 6 – odpowiedź:", odpowiedz_6)

## 5. Częstotliwość próbkowania: ile próbek na sekundę?

Czy przy `fs = 100` (100 próbek na sekundę) możemy zarejestrować dowolnie szybki rytm? **Nie.** Żeby zobaczyć falę, potrzebujemy co najmniej dwóch próbek na każdy jej cykl: jednej „na górce” i jednej „w dołku”. Stąd prosta reguła:

> **Najwyższa częstotliwość, jaką możemy zobaczyć, to połowa częstotliwości próbkowania: `fs / 2`.** Nazywamy ją **częstotliwością Nyquista**.

Dlatego widmo z `np.fft.rfft` zawsze kończy się na `fs / 2` (przy `fs = 100` – na 50 Hz).

A co się dzieje z rytmem szybszym niż `fs / 2`? Nie znika – **udaje wolniejszy**. Zobaczmy falę 9 Hz zapisywaną tylko 10 razy na sekundę.

In [ ]:
t_gesty = np.arange(0, 2, 1 / 1000)           # bardzo gęsta oś czasu – tak wygląda „prawdziwa” fala
prawdziwa = np.sin(2 * np.pi * 9 * t_gesty)   # prawdziwa fala: 9 Hz

fs_rzadkie = 10                               # tylko 10 próbek na sekundę → fs / 2 = 5 Hz, a fala ma 9 Hz!
t_rzadki = np.arange(0, 2, 1 / fs_rzadkie)
probki = np.sin(2 * np.pi * 9 * t_rzadki)     # wartości fali w chwilach pomiaru

fig, ax = plt.subplots()
ax.plot(t_gesty, prawdziwa, color=SZARY, lw=1, label="prawdziwa fala: 9 Hz")
ax.plot(t_rzadki, probki, "o--", color=POMARANCZOWY, ms=8, lw=2, label="to, co zapisał komputer (10 próbek/s)")
ax.set(xlabel="Czas [s]", ylabel="Wartość sygnału", title="Za rzadkie próbkowanie: fala 9 Hz udaje falę 1 Hz",
       ylim=(-1.3, 1.9))                                  # trochę miejsca nad falą na legendę
ax.legend(loc="upper center", ncol=2, framealpha=0.95)   # legenda w jednym wierszu, nad danymi
plt.show()

Pomarańczowe kropki układają się w wolną falę **1 Hz**, której w rzeczywistości nie ma! To zjawisko nazywa się **aliasingiem** (ang. *alias* – fałszywe imię): szybki rytm „podszywa się” pod wolny.

Dlatego urządzenia pomiarowe przed zapisem przepuszczają sygnał przez filtr, który usuwa częstotliwości powyżej `fs / 2`. To pierwszy przykład filtra – więcej o filtrach w części II.

> **Reguła praktyczna:** częstotliwość próbkowania musi być **ponad dwa razy większa** niż najszybszy rytm, który chcemy zarejestrować.

### ✏️ Ćwiczenie 7. Granica Nyquista

Eye-tracker, z którego pochodzą dane w notebooku 01, zapisuje **500 próbek na sekundę** (`fs = 500`). Jaka jest najwyższa częstotliwość, jaką można zobaczyć w jego danych? Policz w pamięci, wpisz odpowiedź, a potem uruchom komórkę – sprawdzi Cię ostatni punkt osi częstotliwości widma.

In [ ]:
# ✏️ Ćwiczenie 7
fs_eyetracker = 500   # częstotliwość próbkowania eye-trackera [Hz]
nyquist = None        # ✏️ wpisz swoją odpowiedź [Hz]

# Sprawdzenie: oś częstotliwości widma dla 1 sekundy nagrania (500 próbek); [-1] = ostatni element
os_czestotliwosci = np.fft.rfftfreq(500, d=1 / fs_eyetracker)
print("Ćwiczenie 7 – Twoja odpowiedź:", nyquist, "Hz")
print("Ostatnia częstotliwość w widmie:", os_czestotliwosci[-1], "Hz")

# Część II · Filtrowanie

## 6. Filtr = wymazywanie części widma

Skoro widmo mówi, z jakich rytmów składa się sygnał, to idea **filtrowania** jest bardzo prosta:

1. liczymy widmo sygnału (transformata Fouriera),
2. **wymazujemy** (zerujemy) częstotliwości, których nie chcemy,
3. składamy sygnał z powrotem – służy do tego **odwrotna transformata Fouriera** (`np.fft.irfft`).

W języku kuchni: odczytujemy przepis, wykreślamy z niego jeden składnik i gotujemy potrawę jeszcze raz.

Najpierw funkcja pomocnicza, która porówna sygnał **przed** i **po** filtrowaniu – w czasie i w widmie.

In [ ]:
def porownaj_przed_i_po(t, przed, po, fs, tytul="", fmax=None, tmax=None):
    """Rysuje sygnał (po lewej) i widmo (po prawej): PRZED filtrowaniem na szaro, PO – na niebiesko."""
    f_przed, A_przed = policz_widmo(przed, fs)
    f_po, A_po = policz_widmo(po, fs)
    fig, (ax_czas, ax_widmo) = plt.subplots(1, 2, figsize=(13, 3.5))

    # Lewy wykres: sygnał w czasie
    ax_czas.plot(t, przed, color=SZARY, lw=1, label="przed")
    ax_czas.plot(t, po, color=NIEBIESKI, lw=2, label="po")
    ax_czas.set(xlabel="Czas [s]", ylabel="Wartość sygnału", title="Sygnał w czasie",
                xlim=(0, tmax if tmax else t[-1]))

    # Prawy wykres: widmo – szerokie szare kreski (przed), na nich wąskie niebieskie (po)
    ax_widmo.vlines(f_przed, 0, A_przed, color=SZARY, lw=6, label="przed")
    ax_widmo.vlines(f_po, 0, A_po, color=NIEBIESKI, lw=2, label="po")
    ax_widmo.set(xlabel="Częstotliwość [Hz]", ylabel="Amplituda", title="Widmo",
                 xlim=(0, fmax if fmax else fs / 2), ylim=(0, None))
    ax_widmo.legend(loc="upper right", framealpha=0.95)

    fig.suptitle(tytul, fontweight="bold")
    plt.tight_layout()
    plt.show()

Teraz sygnał „wolna fala 2 Hz + szybkie drżenie 30 Hz”. Chcemy zostawić wolną falę, a usunąć drżenie, więc wymażemy z widma wszystko powyżej 10 Hz.

In [ ]:
fs = 200                                     # 200 próbek na sekundę
t = np.arange(0, 2, 1 / fs)                  # 2 sekundy
wolna = 1.0 * np.sin(2 * np.pi * 2 * t)      # to chcemy zachować: 2 Hz
drzenie = 0.4 * np.sin(2 * np.pi * 30 * t)   # to chcemy usunąć: 30 Hz
sygnal = wolna + drzenie

granica = 10   # wymażemy wszystkie częstotliwości POWYŻEJ tej granicy [Hz]

wynik_fft = np.fft.rfft(sygnal)                           # 1. sygnał → widmo
czestotliwosci = np.fft.rfftfreq(len(sygnal), d=1 / fs)   #    oś częstotliwości widma
wynik_fft[czestotliwosci > granica] = 0                   # 2. zerujemy wszystko powyżej granicy
sygnal_po = np.fft.irfft(wynik_fft, n=len(sygnal))        # 3. widmo → sygnał (odwrotna transformata)

porownaj_przed_i_po(t, sygnal, sygnal_po, fs, tytul=f"Wymazujemy z widma wszystko powyżej {granica} Hz", fmax=40)

Drżenie zniknęło – została czysta wolna fala. W widmie szary pik przy 30 Hz (przed) nie ma już niebieskiego odpowiednika (po), a pik przy 2 Hz został nietknięty.

> W praktyce nie wymazujemy widma ręcznie. Takie „ostre cięcie” przy nagłych zmianach sygnału tworzy sztuczne falowanie, więc korzystamy z gotowych filtrów, które tną łagodniej (sekcja 7). **Idea jest jednak dokładnie ta sama: filtr usuwa z widma wybrane częstotliwości.**

### ✏️ Ćwiczenie 8. Przewiduj, potem sprawdź

Zanim uruchomisz komórkę, **zgadnij**: co zostanie z sygnału, jeśli ustawimy `granica = 1`, czyli wymażemy wszystko powyżej 1 Hz?

* **A** – nic się nie zmieni,
* **B** – zostanie tylko wolna fala 2 Hz,
* **C** – zostanie płaska linia (prawie zero).

Najpierw wpisz swoją odpowiedź, potem zmień granicę i sprawdź na wykresie.

In [ ]:
# ✏️ Ćwiczenie 8
odpowiedz_8 = None   # ✏️ najpierw zgadnij: "A", "B" albo "C"

granica = 10   # ✏️ zmień na 1 i uruchom komórkę

wynik_fft = np.fft.rfft(sygnal)                           # sygnał → widmo
czestotliwosci = np.fft.rfftfreq(len(sygnal), d=1 / fs)   # oś częstotliwości widma
wynik_fft[czestotliwosci > granica] = 0                   # zerujemy wszystko powyżej granicy
sygnal_po = np.fft.irfft(wynik_fft, n=len(sygnal))        # widmo → sygnał

porownaj_przed_i_po(t, sygnal, sygnal_po, fs, tytul=f"Ćwiczenie 8: wymazujemy powyżej {granica} Hz", fmax=40)
print("Ćwiczenie 8 – odpowiedź:", odpowiedz_8)

## 7. Cztery rodzaje filtrów

Filtry nazywamy według tego, **co przepuszczają**:

| Filtr | Przepuszcza | Usuwa | Przykład użycia |
|---|---|---|---|
| **dolnoprzepustowy** (*low-pass*) | częstotliwości **poniżej** granicy | wysokie (szybkie) | wygładzenie szybkiego szumu |
| **górnoprzepustowy** (*high-pass*) | częstotliwości **powyżej** granicy | niskie (wolne) | usunięcie powolnego „pływania” sygnału (dryfu) |
| **pasmowoprzepustowy** (*band-pass*) | tylko **pasmo** między dwiema granicami | wszystko poza pasmem | wybranie jednego rytmu |
| **pasmowozaporowy**, czyli ***notch*** | wszystko **oprócz** wąskiego pasma | jedną częstotliwość | usunięcie zakłócenia 50 Hz z sieci elektrycznej |

> 🧠 **Jak zapamiętać nazwy?** Nazwa mówi, **co przechodzi** przez filtr, a nie co usuwamy. *Dolno*przepustowy = *przepuszcza dolne* (niskie) częstotliwości. *Górno*przepustowy = *przepuszcza górne* (wysokie). To bardzo częsta pomyłka!

Znasz to z życia: **korektor dźwięku** (*equalizer*) w odtwarzaczu muzyki to zestaw filtrów. „Więcej basów” wzmacnia niskie częstotliwości, „mniej sopranów” osłabia wysokie.

Gotowe filtry daje biblioteka **SciPy**. Zamknęliśmy je w czterech funkcjach o polskich nazwach. Każda ma dwie linijki: zaprojektuj filtr i zastosuj go do sygnału.

In [ ]:
def filtr_dolnoprzepustowy(sygnal, fs, granica):
    """Przepuszcza częstotliwości PONIŻEJ granicy [Hz], usuwa wyższe."""
    sos = signal.butter(4, granica, btype="lowpass", fs=fs, output="sos")   # projekt filtra (filtr Butterwortha)
    return signal.sosfiltfilt(sos, sygnal)   # filtrowanie w przód i w tył – sygnał nie przesuwa się w czasie


def filtr_gornoprzepustowy(sygnal, fs, granica):
    """Przepuszcza częstotliwości POWYŻEJ granicy [Hz], usuwa niższe."""
    sos = signal.butter(4, granica, btype="highpass", fs=fs, output="sos")
    return signal.sosfiltfilt(sos, sygnal)


def filtr_pasmowoprzepustowy(sygnal, fs, od, do):
    """Przepuszcza tylko pasmo OD–DO [Hz], usuwa wszystko poza nim."""
    sos = signal.butter(4, [od, do], btype="bandpass", fs=fs, output="sos")
    return signal.sosfiltfilt(sos, sygnal)


def filtr_notch(sygnal, fs, czestotliwosc):
    """Usuwa jedną wąską częstotliwość [Hz] (np. 50 Hz z sieci), resztę przepuszcza."""
    b, a = signal.iirnotch(czestotliwosc, Q=30, fs=fs)   # Q = 30: wycina pasmo o szerokości ok. czestotliwosc / 30
    return signal.filtfilt(b, a, sygnal)                 # filtrowanie w przód i w tył


print("Cztery filtry gotowe do użycia.")

Liczby `4` (rząd filtra) i `Q=30` mówią, jak „stromo” filtr tnie. Nie musisz ich zmieniać.

### Sygnał testowy: pomiar z czujnika

Zbudujmy sygnał podobny do tego, jaki często daje prawdziwy czujnik. Ma trzy składniki:

* **dryf** 0,5 Hz (amplituda 2) – powolne „pływanie” sygnału, np. przez zmianę temperatury czy ruch kabla,
* **rytm** 8 Hz (amplituda 1) – to, co nas interesuje,
* **zakłócenie z sieci** 50 Hz (amplituda 0,5) – „buczenie” przewodów elektrycznych w budynku. Napięcie w gniazdkach w Polsce zmienia się z częstotliwością 50 Hz.

In [ ]:
fs = 500                                   # 500 próbek na sekundę
t = np.arange(0, 4, 1 / fs)                # 4 sekundy

dryf = 2.0 * np.sin(2 * np.pi * 0.5 * t)   # 0,5 Hz – powolne pływanie
rytm = 1.0 * np.sin(2 * np.pi * 8 * t)     # 8 Hz – to nas interesuje
siec = 0.5 * np.sin(2 * np.pi * 50 * t)    # 50 Hz – zakłócenie z sieci elektrycznej
pomiar = dryf + rytm + siec

rysuj_sygnal_i_widmo(t, pomiar, fs, tytul="Pomiar z czujnika: dryf + rytm + zakłócenie z sieci", fmax=60)

W widmie są trzy piki: 0,5 Hz (wysokość 2), 8 Hz (1) i 50 Hz (0,5). Teraz użyjemy po kolei każdego z czterech filtrów. Na wykresach **szare = przed** filtrowaniem, **niebieskie = po**.

#### Filtr dolnoprzepustowy 20 Hz

Przepuszcza wszystko poniżej 20 Hz → powinien usunąć zakłócenie 50 Hz, a zostawić dryf i rytm.

In [ ]:
po_dolnoprzepustowym = filtr_dolnoprzepustowy(pomiar, fs, granica=20)   # zostaje to, co poniżej 20 Hz
porownaj_przed_i_po(t, pomiar, po_dolnoprzepustowym, fs, tytul="Filtr dolnoprzepustowy 20 Hz", fmax=60)

Linia stała się gładka – drobne „włoski” (50 Hz) zniknęły. Sygnał nadal jednak pływa w górę i w dół, bo dryf (0,5 Hz) jest poniżej granicy i przeszedł przez filtr.

#### Filtr górnoprzepustowy 2 Hz

Przepuszcza wszystko powyżej 2 Hz → powinien usunąć dryf.

In [ ]:
po_gornoprzepustowym = filtr_gornoprzepustowy(pomiar, fs, granica=2)   # zostaje to, co powyżej 2 Hz
porownaj_przed_i_po(t, pomiar, po_gornoprzepustowym, fs, tytul="Filtr górnoprzepustowy 2 Hz", fmax=60)

Sygnał przestał pływać i trzyma się wokół zera. Zakłócenie 50 Hz zostało – jest powyżej granicy.

#### Filtr pasmowoprzepustowy 2–20 Hz

Przepuszcza tylko pasmo 2–20 Hz → usunie i dryf, i zakłócenie. Powinien zostać sam rytm 8 Hz.

In [ ]:
po_pasmowym = filtr_pasmowoprzepustowy(pomiar, fs, od=2, do=20)   # zostaje tylko pasmo 2–20 Hz
porownaj_przed_i_po(t, pomiar, po_pasmowym, fs, tytul="Filtr pasmowoprzepustowy 2–20 Hz", fmax=60)

Został czysty rytm 8 Hz – dokładnie ten składnik, który nas interesował.

#### Filtr notch 50 Hz

Wycina tylko wąskie pasmo wokół 50 Hz → usunie zakłócenie z sieci, a dryf i rytm zostawi.

In [ ]:
po_notch = filtr_notch(pomiar, fs, czestotliwosc=50)   # wycinamy tylko 50 Hz
porownaj_przed_i_po(t, pomiar, po_notch, fs, tytul="Filtr notch 50 Hz", fmax=60)

Ten sam pomiar, cztery filtry – cztery różne wyniki. **Wybór filtra zależy od tego, co chcemy zachować, a co jest dla nas zakłóceniem.**

### ✏️ Ćwiczenie 9. Zostaw tylko zakłócenie

Czasem chcemy obejrzeć samo zakłócenie, np. żeby sprawdzić, jak jest silne. Ustaw granice filtra pasmowoprzepustowego tak, żeby z pomiaru zostało **tylko** buczenie 50 Hz, a zniknęły dryf i rytm. Wynik sprawdź na wykresie i w liczbach pod nim.

In [ ]:
# ✏️ Ćwiczenie 9
od = 2    # ✏️ zmień dolną granicę pasma [Hz]
do = 20   # ✏️ zmień górną granicę pasma [Hz] (musi być mniejsza niż fs / 2 = 250 Hz)

tylko_siec = filtr_pasmowoprzepustowy(pomiar, fs, od=od, do=do)
porownaj_przed_i_po(t, pomiar, tylko_siec, fs, tytul=f"Ćwiczenie 9: pasmo {od}–{do} Hz", fmax=60)

# Wysokość widma po filtrze przy trzech częstotliwościach – powinna zostać tylko ta przy 50 Hz
print("Ćwiczenie 9 – amplitudy po filtrze:")
for f_spr in [0.5, 8, 50]:
    print(f"   {f_spr:>4} Hz: {amplituda_przy(tylko_siec, fs, f_spr):.2f}")

### 🔊 Filtr w akcji: posłuchaj

Wróćmy do dźwięku z sekcji 3 (tony 220, 440 i 880 Hz). Filtr dolnoprzepustowy 600 Hz usunie ton wysoki, a filtr górnoprzepustowy 300 Hz – ton niski. Posłuchaj wszystkich trzech wersji.

In [ ]:
bez_wysokiego = filtr_dolnoprzepustowy(akord, fs_audio, granica=600)   # zostają tony 220 i 440 Hz
bez_niskiego = filtr_gornoprzepustowy(akord, fs_audio, granica=300)    # zostają tony 440 i 880 Hz

porownaj_przed_i_po(t_audio, akord, bez_wysokiego, fs_audio, tytul="Dolnoprzepustowy 600 Hz: znika ton wysoki",
                    fmax=1200, tmax=0.02)
porownaj_przed_i_po(t_audio, akord, bez_niskiego, fs_audio, tytul="Górnoprzepustowy 300 Hz: znika ton niski",
                    fmax=1200, tmax=0.02)

print("Oryginał (trzy tony):")
display(Audio(akord, rate=fs_audio))
print("Po filtrze dolnoprzepustowym – bez tonu wysokiego, brzmi „ciemniej”:")
display(Audio(bez_wysokiego, rate=fs_audio))
print("Po filtrze górnoprzepustowym – bez tonu niskiego, brzmi „jaśniej” i „cieniej”:")
display(Audio(bez_niskiego, rate=fs_audio))

## 8. Jak dobrać filtr?

Dobór filtra to zawsze dwa pytania:

1. **Co chcę usunąć?** Wolne zmiany, szybkie zmiany, a może jedną konkretną częstotliwość?
2. **Gdzie postawić granicę?** Tak, żeby to, co chcę zachować, było po „dobrej stronie”.

Zawsze zaczynaj od widma: **najpierw zobacz, co jest w sygnale, potem dobierz filtr, a na koniec sprawdź widmo po filtrze.**

### ✏️ Ćwiczenie 10. Który filtr?

Dla każdej sytuacji wybierz filtr:

* **A** – dolnoprzepustowy,
* **B** – górnoprzepustowy,
* **C** – pasmowoprzepustowy,
* **D** – notch (pasmowozaporowy).

1. Sygnał przez całe nagranie powoli „odpływa” w górę (dryf). Chcę usunąć to powolne pływanie, a resztę zostawić.
2. Nagranie ma drobny, szybki szum („puszysta” linia). Chcę go wygładzić, a zostawić wolne zmiany.
3. W nagraniu z laboratorium widać pik przy 50 Hz. Chcę usunąć tylko tę jedną częstotliwość.
4. Interesuje mnie tylko rytm między 8 a 12 Hz. Wszystko inne mi przeszkadza.
5. Chcę naraz usunąć wolny dryf (poniżej 1 Hz) i szybki szum (powyżej 40 Hz).

In [ ]:
# ✏️ Ćwiczenie 10 – wpisz literę w cudzysłowie, np. "A"
sytuacja_1 = None   # ✏️ dryf (powolne pływanie)
sytuacja_2 = None   # ✏️ szybki szum
sytuacja_3 = None   # ✏️ zakłócenie 50 Hz
sytuacja_4 = None   # ✏️ tylko rytm 8–12 Hz
sytuacja_5 = None   # ✏️ dryf poniżej 1 Hz i szum powyżej 40 Hz naraz
print("Ćwiczenie 10:", sytuacja_1, sytuacja_2, sytuacja_3, sytuacja_4, sytuacja_5)

## 9. Granica filtra nie jest ostra

Filtr dolnoprzepustowy 20 Hz nie działa jak nóż, który przy 19,9 Hz przepuszcza wszystko, a przy 20,1 Hz – nic. Wokół granicy jest **łagodne zbocze**: częstotliwości blisko granicy przechodzą częściowo.

Sprawdźmy to doświadczalnie. Puśćmy przez ten filtr sinusoidy o częstotliwościach od 1 do 60 Hz (każda ma amplitudę 1) i zmierzmy, ile zostanie z każdej z nich.

In [ ]:
fs = 500                     # 500 próbek na sekundę
t = np.arange(0, 1, 1 / fs)  # 1 sekunda
granica = 20                 # filtr dolnoprzepustowy 20 Hz

lista_f = np.arange(1, 61)   # częstotliwości do sprawdzenia: 1, 2, …, 60 Hz
procent_zostalo = []         # tu zapiszemy wyniki

for f_test in lista_f:
    sinus = np.sin(2 * np.pi * f_test * t)                     # sinusoida o amplitudzie 1
    po_filtrze = filtr_dolnoprzepustowy(sinus, fs, granica)    # przepuszczamy ją przez filtr
    procent = 100 * amplituda_przy(po_filtrze, fs, f_test)     # ile % amplitudy zostało
    procent_zostalo.append(procent)                            # dopisujemy wynik do listy

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lista_f, procent_zostalo, "o-", color=NIEBIESKI, ms=4)
ax.axvline(granica, color=POMARANCZOWY, ls="--", lw=2, label=f"granica filtra: {granica} Hz")
ax.set(xlabel="Częstotliwość sinusoidy [Hz]", ylabel="Zostało amplitudy [%]", ylim=(-5, 105),
       title="Ile sinusoidy przechodzi przez filtr dolnoprzepustowy 20 Hz?")
ax.legend(loc="upper right", framealpha=0.95)
plt.show()

Daleko poniżej granicy przechodzi prawie 100%, daleko powyżej – prawie 0%. Pomiędzy jest jednak łagodne przejście.

**Wniosek praktyczny:** jeśli coś przeszkadza przy 25 Hz, filtr z granicą 20 Hz **nie usunie tego całkowicie**. Granicę trzeba postawić dalej od tego, co chcemy usunąć.

### ✏️ Ćwiczenie 11. Odczytaj z wykresu

Ile procent amplitudy przechodzi przez filtr przy **10 Hz**, **20 Hz** (dokładnie na granicy) i **30 Hz**? Odczytaj z wykresu albo wypisz z listy `procent_zostalo`.

*Wskazówka:* `procent_zostalo[9]` to wynik dla 10 Hz. Python liczy od zera, a lista zaczyna się od 1 Hz, więc numer = częstotliwość − 1.

In [ ]:
# ✏️ Ćwiczenie 11
proc_10 = None   # ✏️ ile % przy 10 Hz
proc_20 = None   # ✏️ ile % przy 20 Hz
proc_30 = None   # ✏️ ile % przy 30 Hz
print(f"Ćwiczenie 11: 10 Hz → {proc_10}%, 20 Hz → {proc_20}%, 30 Hz → {proc_30}%")

## Podsumowanie

* **Sygnał cyfrowy** to ciąg próbek. `fs` to liczba próbek na sekundę.
* **Sinusoidę** opisują **częstotliwość** (ile cykli na sekundę, Hz) i **amplituda** (wysokość fali).
* Złożone sygnały to **sumy sinusoid**.
* **Transformata Fouriera** (FFT) zamienia sygnał w czasie na **widmo**. Położenie piku = częstotliwość, wysokość piku = amplituda.
* W widmie widać rytmy ukryte w szumie. Dłuższe nagranie = czystsze widmo.
* Widzimy tylko częstotliwości do **fs / 2** (częstotliwość Nyquista). Szybsze rytmy udają wolniejsze (aliasing).
* **Filtr usuwa wybrane częstotliwości.** Nazwa mówi, co filtr **przepuszcza**: dolno-, górno- albo pasmowoprzepustowy. Notch wycina jedną częstotliwość.
* Granica filtra nie jest ostra – przy samej granicy przechodzi połowa amplitudy.
* Zasada pracy: **widmo → wybór filtra → widmo po filtrze** (sprawdzenie).

## Słowniczek

| Po polsku | Po angielsku | W jednym zdaniu |
|---|---|---|
| próbka | *sample* | jeden pomiar sygnału |
| częstotliwość próbkowania | *sampling rate*, *sampling frequency* | ile próbek na sekundę (`fs`) |
| częstotliwość | *frequency* | ile cykli fali na sekundę [Hz] |
| amplituda | *amplitude* | wysokość fali |
| widmo | *spectrum* | wykres: jakie częstotliwości są w sygnale i jak silne |
| transformata Fouriera, FFT | *Fourier transform*, *Fast Fourier Transform* | sposób na policzenie widma |
| częstotliwość Nyquista | *Nyquist frequency* | najwyższa widoczna częstotliwość: `fs / 2` |
| aliasing | *aliasing* | szybki rytm udaje wolny, gdy próbkujemy za rzadko |
| filtr dolnoprzepustowy | *low-pass filter* | przepuszcza niskie częstotliwości |
| filtr górnoprzepustowy | *high-pass filter* | przepuszcza wysokie częstotliwości |
| filtr pasmowoprzepustowy | *band-pass filter* | przepuszcza tylko wybrane pasmo |
| filtr pasmowozaporowy, notch | *band-stop filter*, *notch filter* | wycina wąskie pasmo (np. 50 Hz) |
| granica filtra (częstotliwość odcięcia) | *cutoff frequency* | gdzie filtr zaczyna tłumić sygnał |
| dryf | *drift* | powolne „pływanie” sygnału |

## Gdzie to się przyda w kursie?

* **Notebook 01** – średnica źrenicy jest tam wygładzana filtrem **dolnoprzepustowym 4 Hz** (filtr Butterwortha, funkcja `filtfilt`) – to ten sam rodzaj filtra co tutaj. Filtr Savitzky'ego–Golaya, którym wygładza się położenie oka, też działa jak filtr dolnoprzepustowy.
* **Notebook 03** – widmo surowego sygnału, filtr **notch** 50/60 Hz i filtr **pasmowoprzepustowy 1–40 Hz**.
* **Notebook 04** – pasma częstotliwości (np. 8–13 Hz) i analiza czas–częstotliwość, czyli widmo liczone osobno dla kolejnych chwil.

> Biblioteka MNE (notebooki 03–04) rysuje zwykle widmo **mocy** (amplituda do kwadratu) w skali **decybeli** (logarytmicznej). Wykres wygląda inaczej, ale czyta się go tak samo: położenie piku = częstotliwość, wysokość = siła rytmu.

## Zadania na koniec: tętno ze smartwatcha

Smartwatch mierzy tętno światłem. Dioda oświetla skórę nadgarstka, a czujnik mierzy, ile światła wraca. Przy każdym uderzeniu serca naczynia krwionośne lekko się wypełniają i sygnał się zmienia. Na ten rytm nakłada się wolniejszy rytm **oddechu** i **szum** czujnika.

Masz 20-sekundowe nagranie (symulowane, `fs = 100` Hz). Komputer wylosował tętno i tempo oddechu. Twoje zadanie: **odczytać tętno na dwa sposoby** – z widma i z wykresu w czasie – i sprawdzić, czy wyniki się zgadzają.

**Zasady pracy**

1. Najpierw uruchom komórkę z nagraniem (poniżej). Nie zmieniaj jej.
2. Rozwiązanie wpisz w komórce kodu pod treścią zadania. Nowym zmiennym nadawaj nazwy z przedrostkiem zadania: `z1_`, `z2_`, `z3_`.
3. Korzystaj z funkcji z notebooka: `rysuj_sygnal_i_widmo`, `amplituda_przy`, `filtr_pasmowoprzepustowy`, `porownaj_przed_i_po`.
4. **Kontrola pośrednia** to wartość, którą możesz sprawdzić po drodze.
5. **✅ Wynik** to jedna linijka do wypisania na końcu, w podanym formacie.

In [ ]:
# Nagranie ze smartwatcha (symulowane) – uruchom tę komórkę, ale jej nie zmieniaj
fs_zegarek = 100                               # 100 próbek na sekundę
t_zegarek = np.arange(0, 20, 1 / fs_zegarek)   # 20 sekund nagrania

losowanie = np.random.default_rng(9)                         # stałe ziarno – za każdym razem ten sam „los”
tetno_tajne = losowanie.choice([60, 66, 72, 78, 84])         # losowe tętno [uderzeń na minutę] – nie podglądaj!
oddech_tajny = losowanie.choice([12, 15, 18])                # losowe tempo oddechu [oddechów na minutę]

oddech = 1.0 * np.sin(2 * np.pi * oddech_tajny / 60 * t_zegarek)    # rytm oddechu (na minutę / 60 = Hz)
serce = 0.5 * np.sin(2 * np.pi * tetno_tajne / 60 * t_zegarek)      # rytm serca
szum_czujnika = 0.3 * losowanie.standard_normal(len(t_zegarek))     # szum czujnika
zegarek = oddech + serce + szum_czujnika                            # to „widzi” smartwatch

fig, ax = plt.subplots()
ax.plot(t_zegarek, zegarek, color="black", lw=1)
ax.set(xlabel="Czas [s]", ylabel="Wartość sygnału", title="Nagranie ze smartwatcha (20 s)")
plt.show()

### Zadanie 1. Tętno z widma

*Powtarzasz: sekcje 3–4 (widmo, odczytywanie pików).*

**Krok po kroku:**

1. Narysuj sygnał i widmo nagrania funkcją `rysuj_sygnal_i_widmo`. Argumenty: `t_zegarek`, `zegarek`, `fs_zegarek`, a do tego `fmax=3` – interesują nas tylko wolne rytmy.
2. W widmie zobaczysz dwa wyraźne piki. **Wolniejszy** (bliżej zera) to oddech, **szybszy** to serce.
3. Odczytaj częstotliwość piku serca i zapisz ją w zmiennej `z1_f_serca`.
4. Przelicz ją na **uderzenia na minutę**: `z1_tetno = z1_f_serca * 60` (Hz to uderzenia na sekundę, a minuta ma 60 sekund).

*Wskazówka:* jeśli trudno odczytać dokładną wartość, zmniejsz `fmax` do 2 albo sprawdź swoją odpowiedź funkcją `amplituda_przy` – przy właściwej częstotliwości wynik będzie bliski 0,5.

**Kontrola pośrednia:** pik oddechu leży przy 0,3 Hz, czyli 0,3 · 60 = 18 oddechów na minutę.

**✅ Wynik:** `print(f"Zadanie 1: serce {z1_f_serca} Hz, tętno = {z1_tetno:.0f} na minutę")`

**Do przemyślenia:** jakie tętno ma osoba w spoczynku? Czy Twój wynik jest realistyczny?

In [ ]:
# Zadanie 1 – Twoje rozwiązanie



# print(f"Zadanie 1: serce {z1_f_serca} Hz, tętno = {z1_tetno:.0f} na minutę")

### Zadanie 2. Oczyść nagranie filtrem

*Powtarzasz: sekcje 7–8 (filtr pasmowoprzepustowy, porównanie przed i po).*

**Cel:** zostawić sam rytm serca – usunąć wolniejszy oddech i szybki szum.

**Krok po kroku:**

1. Przepuść nagranie `zegarek` przez `filtr_pasmowoprzepustowy` z pasmem **od 0,7 do 3 Hz**. Wynik zapisz w zmiennej `z2_czysty`.
   * Dlaczego 0,7 Hz? Ta granica leży między oddechem (0,3 Hz) a sercem.
   * Dlaczego 3 Hz? Serce prawie nigdy nie bije szybciej niż 180 razy na minutę, czyli 3 Hz. Wszystko powyżej to szum.
2. Narysuj porównanie funkcją `porownaj_przed_i_po` (argumenty: `t_zegarek`, `zegarek`, `z2_czysty`, `fs_zegarek`; ustaw `fmax=5`).
3. Zmierz funkcją `amplituda_przy` amplitudę oddechu (przy 0,3 Hz) **przed** filtrem (`z2_przed`) i **po** filtrze (`z2_po`).

**Kontrola pośrednia:** amplituda przy częstotliwości serca po filtrze jest prawie taka sama jak przed (ok. 0,5).

**✅ Wynik:** `print(f"Zadanie 2: amplituda oddechu przed = {z2_przed:.2f}, po = {z2_po:.2f}")`

In [ ]:
# Zadanie 2 – Twoje rozwiązanie



# print(f"Zadanie 2: amplituda oddechu przed = {z2_przed:.2f}, po = {z2_po:.2f}")

### Zadanie 3. Tętno z wykresu w czasie

*Powtarzasz: sekcja 1 (liczenie górek), sekcja 2 (wykres w czasie).*

**Krok po kroku:**

1. Narysuj oczyszczony sygnał `z2_czysty` w czasie (tak jak wykresy w sekcji 1), ale tylko **pierwsze 10 sekund**: `ax.set_xlim(0, 10)`. Tytuł wykresu zacznij od „Zadanie 3”.
2. Policz górki, czyli uderzenia serca, w ciągu tych 10 sekund. Wynik zapisz w `z3_uderzenia`.
3. Przelicz na minutę: `z3_tetno = z3_uderzenia * 6` (minuta to 6 razy po 10 sekund).

**Kontrola pośrednia:** wynik z punktu 3 powinien być taki sam jak tętno z zadania 1. Jeśli nie jest – policz górki jeszcze raz!

**✅ Wynik:** `print(f"Zadanie 3: {z3_uderzenia} uderzeń w 10 s, tętno = {z3_tetno} na minutę")`

**Do przemyślenia:** spróbuj policzyć górki na **surowym** nagraniu `zegarek`. Dlaczego to dużo trudniejsze?

In [ ]:
# Zadanie 3 – Twoje rozwiązanie



# print(f"Zadanie 3: {z3_uderzenia} uderzeń w 10 s, tętno = {z3_tetno} na minutę")

## Literatura

* Smith, S. W. (1997). *The scientist and engineer's guide to digital signal processing*. California Technical Publishing. Książka dostępna bezpłatnie: <https://www.dspguide.com>
* Zieliński, T. P. (2005). *Cyfrowe przetwarzanie sygnałów. Od teorii do zastosowań*. Wydawnictwa Komunikacji i Łączności.
* Cohen, M. X. (2014). *Analyzing neural time series data: Theory and practice*. MIT Press. (rozdz. 11 – transformata Fouriera, rozdz. 14 – filtrowanie)
* Widmann, A., Schröger, E., i Maess, B. (2015). Digital filter design for electrophysiological data – a practical approach. *Journal of Neuroscience Methods, 250*, 34–46. (dla chętnych)
* 3Blue1Brown (2018). *But what is the Fourier Transform? A visual introduction* [film, ang.]. <https://www.youtube.com/watch?v=spUNpyF58BY>